In [ ]:
# %% ─── CELL 1: SETUP + SHARED PREPROCESSING ──────────────────────────────────
import os, re, glob, time, random, hashlib, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = (256, 256)
CROP     = 0.05
CLASSES  = ["NORMAL", "PNEUMONIA"]

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_gray(g):
    """THE preprocessing. Training, validation, external tests must all use this."""
    h, w = g.shape
    dy, dx = int(h * CROP), int(w * CROP)
    g = g[dy:h - dy, dx:w - dx]
    g = cv2.resize(g, IMG_SIZE, interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(g, (1, 99))
    g = np.clip((g.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
    g = (g * 255).astype(np.uint8)
    return clahe.apply(g)

def preprocess_file(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return None if g is None else preprocess_gray(g)

def md5_bytes(b):
    return hashlib.md5(b).hexdigest()

def build_split(files, labels, patient_ids, out_root, seed=SEED):
    """Patient-level split, md5-dedup, write neutral-name PNGs."""
    seen = {}
    for f, lab, pid in zip(files, labels, patient_ids):
        a = preprocess_file(f)
        if a is None:
            continue
        h = md5_bytes(a.tobytes())
        if h in seen:
            continue
        seen[h] = (a, lab, pid)
    items = list(seen.values())
    print("after md5 dedup:", len(items))
    Xpid = np.array([i[2] for i in items])
    y = np.array([i[1] for i in items])
    gs = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    tr_idx, te_idx = next(gs.split(np.zeros(len(items)), y, groups=Xpid))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=seed)  # 0.15/0.85
    tr_idx2, va_idx2 = next(gss2.split(np.zeros(len(tr_idx)), y[tr_idx], groups=Xpid[tr_idx]))
    splits = {"train": tr_idx[tr_idx2], "val": tr_idx[va_idx2], "test": te_idx}
    for split, idxs in splits.items():
        os.makedirs(os.path.join(out_root, split), exist_ok=True)
        for c in CLASSES:
            os.makedirs(os.path.join(out_root, split, c), exist_ok=True)
    for split, idxs in splits.items():
        rng = random.Random(seed)
        rng.shuffle(idxs)
        ctr = {c: 0 for c in CLASSES}
        for i in idxs:
            img, lab, _ = items[i]
            c = CLASSES[lab]
            cv2.imwrite(os.path.join(out_root, split, c, f"{c}_{ctr[c]}.png"), img)
            ctr[c] += 1
    return {s: {c: len(os.listdir(os.path.join(out_root, s, c))) for c in CLASSES} for s in splits}

# find Kermany root
cands = []
for pat in ["/kaggle/input/*/chest_xray", "/kaggle/input/*/*/chest_xray", "/kaggle/input/*/*/*/chest_xray"]:
    cands += [p for p in glob.glob(pat) if os.path.isdir(os.path.join(p, "train"))]
assert cands, "Kermany chest_xray folder not found"
KROOT = cands[0]
print("Kermany root:", KROOT)

def kermany_patient_id(cls, fname):
    m = re.match(r"person(\d+)_", fname) if cls == "PNEUMONIA" else re.search(r"IM-(\d+)", fname)
    return cls + (m.group(1) if m else fname)

kfiles, klabels, kpids = [], [], []
for split in ["train", "val", "test"]:
    for ci, c in enumerate(CLASSES):
        d = os.path.join(KROOT, split, c)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                kfiles.append(os.path.join(d, f))
                klabels.append(ci)
                kpids.append(kermany_patient_id(c, f))
KERMANY_CLEAN = "/kaggle/working/clean_kermany"
kcounts = build_split(kfiles, klabels, kpids, KERMANY_CLEAN)
print("Kermany split:", kcounts)


In [ ]:
# %% ─── CELL 2: NIH SAMPLE LOAD + PATIENT-LEVEL SPLIT ────────────────────────
# NIH Chest X-ray14 "Random Sample" (nih-chest-xrays/sample). Labels are NLP-mined;
# we consolidate pneumonia-like findings into one positive class.
import pandas as pd

csvs = glob.glob("/kaggle/input/**/sample_labels.csv", recursive=True)
assert csvs, "sample_labels.csv not found — attach nih-chest-xrays/sample"
NIH_CSV = csvs[0]
print("NIH csv:", NIH_CSV)

ndf = pd.read_csv(NIH_CSV)
print("NIH rows:", len(ndf))
print("labels sample:", ndf["Finding Labels"].iloc[0])

POS = {"Pneumonia", "Consolidation", "Infiltration"}

def nih_label(lbl):
    parts = [p.strip() for p in str(lbl).split("|")]
    if "No Finding" in parts and len(parts) == 1:
        return 0
    if any(p in POS for p in parts):
        return 1
    return None  # other single/multi-label findings: skip

img_paths = {}
for pat in ["/kaggle/input/**/*.png", "/kaggle/input/**/*.jpg", "/kaggle/input/**/*.jpeg"]:
    for p in glob.glob(pat, recursive=True):
        img_paths[os.path.basename(p)] = p
print("NIH image files found:", len(img_paths))

nfiles, nlabels, npids = [], [], []
for _, row in ndf.iterrows():
    lab = nih_label(row["Finding Labels"])
    if lab is None:
        continue
    name = row["Image Index"]
    p = img_paths.get(name)
    if p:
        nfiles.append(p)
        nlabels.append(lab)
        npids.append(str(row.get("Patient ID", name)))
print("NIH usable:", len(nfiles), "| positives:", sum(nlabels), "| negatives:", len(nlabels) - sum(nlabels))
NIH_CLEAN = "/kaggle/working/clean_nih"
ncounts = build_split(nfiles, nlabels, npids, NIH_CLEAN)
print("NIH split:", ncounts)


In [ ]:
# %% ─── CELL 3: MERGE KERMANY + NIH TRAIN/VAL VIA SYMLINKS ───────────────────
MERGED = "/kaggle/working/clean_merged"
merged_counts = {}
for split in ["train", "val"]:  # tests stay separate
    for c in CLASSES:
        dst = os.path.join(MERGED, split, c)
        os.makedirs(dst, exist_ok=True)
        n = 0
        for src_root in [KERMANY_CLEAN, NIH_CLEAN]:
            sdir = os.path.join(src_root, split, c)
            for f in os.listdir(sdir):
                src = os.path.join(sdir, f)
                tag = "kermany" if "clean_kermany" in sdir else "nih"
                dstf = os.path.join(dst, f"{tag}_{f}")
                try:
                    if not os.path.exists(dstf):
                        os.symlink(src, dstf)
                except OSError:
                    shutil.copy2(src, dstf)
                n += 1
        merged_counts[f"{split}/{c}"] = n
print("Merged:", merged_counts)


In [ ]:
# %% ─── CELL 4: GENERATORS + CLASS WEIGHTS (MERGED TRAIN, MERGED VAL) ────────
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=8, zoom_range=0.08, width_shift_range=0.04, height_shift_range=0.04)
val_datagen   = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(os.path.join(MERGED, "train"), target_size=IMG_SIZE, color_mode="rgb", class_mode="binary", classes=CLASSES, batch_size=32, shuffle=True, seed=SEED)
val_gen   = val_datagen.flow_from_directory(os.path.join(MERGED, "val"),   target_size=IMG_SIZE, color_mode="rgb", class_mode="binary", classes=CLASSES, batch_size=32, shuffle=False)

k_test_gen  = val_datagen.flow_from_directory(os.path.join(KERMANY_CLEAN, "test"), target_size=IMG_SIZE, color_mode="rgb", class_mode="binary", classes=CLASSES, batch_size=32, shuffle=False)
n_test_gen  = val_datagen.flow_from_directory(os.path.join(NIH_CLEAN, "test"),     target_size=IMG_SIZE, color_mode="rgb", class_mode="binary", classes=CLASSES, batch_size=32, shuffle=False)

class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=train_gen.classes)
class_weights = dict(enumerate(class_weights))
print("Merged class weights:", class_weights)


In [ ]:
# %% ─── CELL 5: DENSENET121 + 2-PHASE TRAINING (MERGED) ──────────────────────
base = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base.trainable = False

inp = layers.Input((*IMG_SIZE, 3))
x = layers.Normalization(axis=-1, mean=[0.485, 0.456, 0.406],
                         variance=[0.229**2, 0.224**2, 0.225**2])(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inp, out)

MODEL_PATH = "/kaggle/working/densenet_merged.keras"

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_gen, validation_data=val_gen, epochs=6, class_weight=class_weights)

base.trainable = True
for l in base.layers[:-60]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
cbs = [
    callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]
history = model.fit(train_gen, validation_data=val_gen, epochs=25, callbacks=cbs, class_weight=class_weights)
print("done training; weights saved to", MODEL_PATH)


In [ ]:
# %% ─── CELL 6: EVALUATION ON BOTH HELD-OUT TEST SETS ────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
SAVE_DIR = "/kaggle/working/evaluation_merged"; os.makedirs(SAVE_DIR, exist_ok=True)
model = tf.keras.models.load_model(MODEL_PATH)

for name, gen in [("Kermany test", k_test_gen), ("NIH held-out test", n_test_gen)]:
    yt = gen.classes
    probs = model.predict(gen, verbose=0).ravel()
    pred = (probs > 0.5).astype(int)
    rep = classification_report(yt, pred, target_names=CLASSES, digits=4)
    print(f"\n=== {name} ===\n", rep)
    open(f"{SAVE_DIR}/{name.split()[0].lower()}_report.txt", "w").write(rep)
    print(f"AUC: {roc_auc_score(yt, probs):.4f}")
    ConfusionMatrixDisplay(confusion_matrix(yt, pred), display_labels=CLASSES).plot(cmap="Blues")
    plt.title(f"{name} confusion matrix"); plt.savefig(f"{SAVE_DIR}/{name.split()[0].lower()}_cm.png"); plt.show()
